# 01 — Signal walkthrough

One clip, every intermediate signal plotted. This notebook is three things at
once: the debugging tool, the figure set for the report, and the artefact that
makes the viva easy.

Pipeline under inspection:

```
video ──► ROI ──► spatial mean ──► RESAMPLE ──► detrend ──► project ──► bandpass ──► window ──► FFT ──► peak ──► BPM
                    (R,G,B)      (uniform fs)              (4 arms)                  (Hann)  (0-pad)  (interp)
```

It defaults to a **synthetic clip with a known 72 BPM pulse**, so it runs before
any dataset arrives and every stage can be checked against ground truth. Point
`CLIP` at a real video to run the same cells on real data.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfreqz, lfilter

from rppg.synthetic import synth_rgb, nonuniform_timestamps
from rppg.resample import resample_uniform, frame_interval_stats
from rppg.preprocess import (detrend_smoothness_priors, detrend_moving_average,
                             bandpass, normalize_zscore)
from rppg.methods import PROJECTIONS, ica_full
from rppg.spectral import estimate_bpm, periodogram, bpm_resolution
from rppg.quality import snr_db
from rppg.pipeline import PipelineConfig, analyse_signal, project_and_filter
from rppg.metrics import mae, rmse, pearson

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9,
                     "figure.facecolor": "white"})
COL = {"green": "#2e9e4f", "ica": "#c2410c", "chrom": "#1d4ed8", "pos": "#7c3aed"}

FS = 30.0            # target uniform sampling rate
WINDOW_SEC = 15.0    # 60/15 = 4 BPM raw resolution
TRUE_BPM = 72.0
CLIP = None          # e.g. "../data/own/A_still/subject1.mp4"


## Stage 0 — the clip

The synthetic generator builds RGB means from the same reflection model the
methods assume,

$$C_c(t) = I(t)\big(v_s(t) + v_d(t)\big) + v_n(t)$$

with a 0.4 % pulsatile modulation (realistic for a webcam), a respiration
baseline, sensor noise, and non-uniform capture timestamps.


In [ ]:
if CLIP is None:
    ts = nonuniform_timestamps(60.0, FS, jitter_ms=3.0, drop_prob=0.005, rng=0)
    clip = synth_rgb(timestamps=ts, bpm=TRUE_BPM, pulse_amplitude=0.004,
                     respiration_bpm=15.0, noise_std=0.15, rng=0)
    t_raw, rgb_raw = clip.t, clip.rgb
    reference_bpm = clip.bpm_true
else:
    from rppg.capture import VideoFileSource
    from rppg.roi import make_roi, extract_signal
    with VideoFileSource(CLIP) as src:
        roi = make_roi("mediapipe")
        t_raw, rgb_raw, _ = extract_signal(src, roi)
    reference_bpm = None

print(f"{len(t_raw)} usable frames over {t_raw[-1]:.1f} s")


## Stage 1.5 — capture jitter

The step the original proposal is missing, and the one most likely to silently
ruin the results. Every FFT downstream assumes uniform sampling; a nominal
30 fps that is really a 28.4 fps mean scales the whole frequency axis.


In [ ]:
stats = frame_interval_stats(t_raw)
print(stats)

dt_ms = np.diff(t_raw) * 1e3
fig, ax = plt.subplots(1, 2, figsize=(10, 2.6))
ax[0].hist(dt_ms, bins=40, color="#334155")
ax[0].axvline(1000 / FS, color="#dc2626", ls="--", label=f"nominal {1000/FS:.2f} ms")
ax[0].set(xlabel="frame interval (ms)", ylabel="count", title="capture jitter")
ax[0].legend(fontsize=8)
ax[1].plot(t_raw[1:], dt_ms, lw=0.6, color="#334155")
ax[1].axhline(1000 / FS, color="#dc2626", ls="--")
ax[1].set(xlabel="time (s)", ylabel="dt (ms)", title="interval over time")
fig.tight_layout()


### What assuming uniformity costs

If the true mean rate is $f_{true}$ and we process as if it were $f_{assumed}$,
every frequency is scaled by $f_{assumed}/f_{true}$. Here is that error, made
explicit.


In [ ]:
true_fs = 1.0 / np.mean(np.diff(t_raw))
grid = np.arange(len(rgb_raw)) / FS                      # pretending it is uniform
naive = estimate_bpm(bandpass(detrend_smoothness_priors(rgb_raw[:450, 1]), FS), FS).bpm

t_u, rgb_u = resample_uniform(t_raw, rgb_raw, fs=FS)     # the correct path
fixed = estimate_bpm(bandpass(detrend_smoothness_priors(rgb_u[:450, 1]), FS), FS).bpm

print(f"true mean rate      : {true_fs:.3f} Hz")
print(f"assumed rate        : {FS:.3f} Hz")
print(f"BPM, naive          : {naive:.2f}")
print(f"BPM, resampled      : {fixed:.2f}")
print(f"reference           : {TRUE_BPM:.2f}" if CLIP is None else "")
print(f"uniform grid        : {len(t_u)} samples, dt = {np.diff(t_u)[0]*1e3:.4f} ms exactly")


## Stage 1 — raw RGB, and why it looks hopeless

Three noisy, drifting lines. The pulse is 0.1–1 % of the intensity — it is in
there, but nothing about this plot suggests a heartbeat. That is the honest
starting point.


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
for i, (name, c) in enumerate(zip("RGB", ["#dc2626", "#16a34a", "#2563eb"])):
    ax[0].plot(t_u, rgb_u[:, i], lw=0.7, color=c, label=name)
ax[0].legend(ncol=3, fontsize=8, frameon=False)
ax[0].set(ylabel="mean intensity", title="raw spatial means over the ROI")

zoom = slice(0, int(10 * FS))
g = rgb_u[zoom, 1]
ax[1].plot(t_u[zoom], (g - g.mean()) / g.std(), lw=0.9, color="#16a34a")
ax[1].set(xlabel="time (s)", ylabel="z-score", title="green channel, 10 s, standardised — still dominated by drift")
fig.tight_layout()


## Stage 3 — detrending

Respiration produces a large ~0.2–0.4 Hz baseline wander that the cardiac signal
rides on. A sharp high-pass near that band rings; smoothness-priors detrending
(Tarvainen et al., 2002) is effectively a time-varying high-pass with a very
smooth response.


In [ ]:
raw_g = rgb_u[:, 1]
d_sp = detrend_smoothness_priors(raw_g, lam=100.0)
d_ma = detrend_moving_average(raw_g, FS, win_sec=1.0)

fig, ax = plt.subplots(2, 1, figsize=(10, 4.2))
seg = slice(int(5 * FS), int(20 * FS))
ax[0].plot(t_u[seg], raw_g[seg] - raw_g[seg].mean(), lw=0.8, color="#94a3b8", label="raw (mean removed)")
ax[0].plot(t_u[seg], d_sp[seg], lw=1.0, color="#1d4ed8", label="smoothness priors, λ=100")
ax[0].plot(t_u[seg], d_ma[seg], lw=1.0, color="#ea580c", label="moving average, 1 s")
ax[0].legend(fontsize=8, frameon=False); ax[0].set(xlabel="time (s)", title="detrending, 15 s excerpt")

for lbl, sig, c in [("raw", raw_g, "#94a3b8"), ("priors", d_sp, "#1d4ed8"), ("moving avg", d_ma, "#ea580c")]:
    f, p = periodogram(sig, FS, zero_pad=4)
    m = f <= 4.5
    ax[1].semilogy(f[m] * 60, p[m] + 1e-12, lw=0.9, color=c, label=lbl)
ax[1].axvspan(0.7 * 60, 4.0 * 60, color="#22c55e", alpha=0.07, label="cardiac band")
ax[1].legend(fontsize=8, frameon=False); ax[1].set(xlabel="BPM", ylabel="power", title="spectra — the sub-0.5 Hz wander is what gets removed")
fig.tight_layout()


## Stage 4 — bandpass, and why `filtfilt`

4th-order Butterworth over 0.7–4.0 Hz (42–240 BPM): a resting athlete to
maximal exertion. Applied forward-and-backward, so phase distortion cancels
exactly — at the cost of doubling the effective order and needing the whole
segment. That matters the moment you want beat-to-beat timing.


In [ ]:
sos = butter(4, [0.7 / (FS/2), 4.0 / (FS/2)], btype="bandpass", output="sos")
w, h = sosfreqz(sos, worN=4096, fs=FS)

fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
ax[0].plot(w * 60, 20 * np.log10(np.abs(h) + 1e-12), color="#1d4ed8")
ax[0].axvspan(42, 240, color="#22c55e", alpha=0.08)
ax[0].set(xlim=(0, 400), ylim=(-80, 5), xlabel="BPM", ylabel="dB", title="4th-order Butterworth (one pass)")

b, a = butter(4, [0.7 / (FS/2), 4.0 / (FS/2)], btype="bandpass")
one_way = lfilter(b, a, d_sp)
zero_phase = bandpass(d_sp, FS)
z = slice(int(8 * FS), int(13 * FS))
ax[1].plot(t_u[z], one_way[z] / one_way.std(), lw=1.0, color="#dc2626", label="lfilter (one pass)")
ax[1].plot(t_u[z], zero_phase[z] / zero_phase.std(), lw=1.0, color="#1d4ed8", label="filtfilt (zero phase)")
ax[1].legend(fontsize=8, frameon=False); ax[1].set(xlabel="time (s)", title="the same peaks, shifted in time")

# Measure the delay on a clean symmetric burst, where "where is the peak?" has
# an unambiguous answer — on the pulse signal itself the cross-correlation of
# two near-periodic traces is ambiguous to within a whole cardiac cycle.
tb = np.arange(int(20 * FS)) / FS
burst = np.sin(2*np.pi*1.2*(tb - 10)) * np.exp(-((tb - 10)**2) / 4)
b_one, b_zero = lfilter(b, a, burst), bandpass(burst, FS)
delay = (np.argmax(np.abs(b_one)) - np.argmax(np.abs(b_zero))) / FS * 1000

ax[2].plot(tb, burst / np.abs(burst).max(), lw=0.9, color="#94a3b8", label="input burst")
ax[2].plot(tb, b_zero / np.abs(b_zero).max(), lw=1.0, color="#1d4ed8", label="filtfilt")
ax[2].plot(tb, b_one / np.abs(b_one).max(), lw=1.0, color="#dc2626", label="lfilter")
ax[2].set(xlim=(7, 14), xlabel="time (s)", title=f"lfilter delays the peak by {delay:.0f} ms")
ax[2].legend(fontsize=7, frameon=False)
fig.tight_layout()
print(f"filtfilt peak stays on the input peak; lfilter moves it {delay:.0f} ms later")
print("That delay is why zero-phase filtering matters the moment you want beat-to-beat timing.")


## The four projections

Every arm is one choice of a projection $\mathbf{w}$ such that
$\mathbf{w}^\top[R,G,B]^\top$ suppresses $I(t)v_s(t)$ and keeps the pulsatile
part of $I(t)v_d(t)$. That framing turns four seemingly unrelated algorithms
into one comparable family.

| Arm | Projection | Type |
|---|---|---|
| GREEN | $[0,1,0]$ | fixed, trivial |
| ICA | learned per window | data-driven |
| CHROM | fixed model + adaptive scalar | model-based |
| POS | fixed model + adaptive scalar | model-based |


In [ ]:
cfg = PipelineConfig(fs=FS, window_sec=WINDOW_SEC)
win = rgb_u[: int(WINDOW_SEC * FS)]
signals = {m: project_and_filter(win, FS, m, cfg) for m in PROJECTIONS}

fig, axes = plt.subplots(4, 1, figsize=(10, 5.4), sharex=True)
tw = np.arange(win.shape[0]) / FS
for ax, (name, sig) in zip(axes, signals.items()):
    ax.plot(tw, sig / sig.std(), lw=1.0, color=COL[name])
    ax.set_ylabel(name.upper(), color=COL[name])
    ax.set_yticks([])
axes[0].set_title(f"extracted pulse, {WINDOW_SEC:.0f} s window (amplitude-normalised — "
                  "ICA has scale and sign ambiguity, so only frequency content is usable)")
axes[-1].set_xlabel("time (s)")
fig.tight_layout()


### ICA — what the selector actually does

ICA does not order its outputs. "Component 1 = noise, component 2 =
respiration, component 3 = pulse" is not something the algorithm guarantees —
it is what the selection step decides after the fact, and the ordering changes
with nothing more than the initialisation. Hence the spectral selector below is
mandatory, not a convenience.


In [ ]:
res_ica = ica_full(win, FS)
fig, axes = plt.subplots(3, 1, figsize=(10, 4.2), sharex=True)
for i, ax in enumerate(axes):
    picked = (i == res_ica.chosen)
    ax.plot(tw, res_ica.components[:, i], lw=0.9, color="#c2410c" if picked else "#9ca3af")
    ax.set_ylabel(f"IC{i}"); ax.set_yticks([])
    ax.text(0.995, 0.82, f"spectral concentration {res_ica.scores[i]:.3f}"
            + ("   ← selected" if picked else ""), transform=ax.transAxes,
            ha="right", fontsize=8, color="#c2410c" if picked else "#6b7280")
axes[0].set_title("ICA components — the one the selector chose")
axes[-1].set_xlabel("time (s)")
fig.tight_layout()

print("same data, different FastICA initialisation:")
for rs in range(6):
    r = ica_full(win, FS, random_state=rs)
    print(f"  random_state={rs}: pulse landed on component {r.chosen}  scores={np.round(r.scores,3)}")


## Stage 5 — the FFT resolution problem

Frequency resolution is set by window *duration*, not FFT length:

$$\Delta f = \frac{1}{T} \quad\Longrightarrow\quad \Delta\text{BPM} = \frac{60}{T}$$

| Window | Resolution |
|---|---|
| 5 s | 12 BPM |
| 10 s | 6 BPM |
| 15 s | 4 BPM |
| 20 s | 3 BPM |
| 30 s | 2 BPM |

A 6 BPM quantisation step makes any sub-6-BPM MAE claim meaningless. Zero-padding
**interpolates the spectrum, it does not add resolution** — the demo below shows
both halves of that sentence.


In [ ]:
for T in (5, 10, 15, 20, 30):
    print(f"T = {T:2d} s  ->  {bpm_resolution(T):5.2f} BPM")

# Two tones 3 BPM apart in a 10 s window: padding cannot separate them.
tt = np.arange(int(10 * FS)) / FS
two = np.sin(2*np.pi*1.15*tt) + np.sin(2*np.pi*1.20*tt)   # 69 and 72 BPM
tt2 = np.arange(int(20 * FS)) / FS
two_long = np.sin(2*np.pi*1.15*tt2) + np.sin(2*np.pi*1.20*tt2)

fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for pad, c, lbl in [(1, "#94a3b8", "no padding"), (64, "#1d4ed8", "64× padding")]:
    f, p = periodogram(two, FS, zero_pad=pad)
    m = (f > 1.0) & (f < 1.4)
    ax[0].plot(f[m]*60, p[m]/p[m].max(), lw=1.1, color=c, marker="o", ms=2.5, label=lbl)
ax[0].axvline(69, color="#dc2626", ls=":", lw=1); ax[0].axvline(72, color="#dc2626", ls=":", lw=1)
ax[0].legend(fontsize=8, frameon=False)
ax[0].set(xlabel="BPM", title="10 s window: one peak, however much padding")

f, p = periodogram(two_long, FS, zero_pad=64); m = (f > 1.0) & (f < 1.4)
ax[1].plot(f[m]*60, p[m]/p[m].max(), lw=1.1, color="#16a34a")
ax[1].axvline(69, color="#dc2626", ls=":", lw=1); ax[1].axvline(72, color="#dc2626", ls=":", lw=1)
ax[1].set(xlabel="BPM", title="20 s window: two peaks — resolution came from T")

# But padding + parabolic interpolation does locate a single peak precisely.
single = np.sin(2*np.pi*1.19*tt)   # 71.4 BPM, deliberately between bins
errs = {"raw bin, no pad": estimate_bpm(single, FS, zero_pad=1, interpolate=False).bpm,
        "8× pad, no interp": estimate_bpm(single, FS, zero_pad=8, interpolate=False).bpm,
        "8× pad + parabolic": estimate_bpm(single, FS, zero_pad=8, interpolate=True).bpm}
ax[2].barh(list(errs), [abs(v - 71.4) for v in errs.values()], color=["#dc2626", "#ea580c", "#16a34a"])
ax[2].set(xlabel="|error| (BPM)", title="locating one isolated peak (true 71.4)")
fig.tight_layout()
for k, v in errs.items():
    print(f"{k:>20s}: {v:7.3f} BPM   (error {v-71.4:+.3f})")


### Windowing — Hann vs rectangular

A rectangular window's first sidelobe is at −13 dB; Hann's is at −31 dB, at the
cost of a main lobe twice as wide. On a signal with any residual trend that
difference decides whether the peak you find is the pulse or leakage.


In [ ]:
leaky = 0.02 * np.sin(2*np.pi*1.2*tt) + tt / tt.max()     # tiny pulse on a big ramp
fig, ax = plt.subplots(1, 2, figsize=(10, 2.8))
for wname, c in [(None, "#dc2626"), ("hann", "#16a34a")]:
    f, p = periodogram(leaky, FS, window=wname, zero_pad=8)
    m = (f >= 0.5) & (f <= 4.5)
    ax[0].semilogy(f[m]*60, p[m], lw=1.0, color=c, label=wname or "rectangular")
    est = estimate_bpm(leaky, FS, window=wname)
    ax[1].bar(wname or "rectangular", est.bpm, color=c)
    print(f"{wname or 'rectangular':>12s} window -> {est.bpm:6.2f} BPM")
ax[0].axvline(72, color="k", ls=":", lw=1)
ax[0].legend(fontsize=8, frameon=False); ax[0].set(xlabel="BPM", ylabel="power", title="leakage into the cardiac band")
ax[1].axhline(72, color="k", ls=":", lw=1); ax[1].set(ylabel="estimated BPM", title="…and what it does to the answer")
fig.tight_layout()


## Spectrum per arm, with the peak marked

Plus the de Haan & Jeanne SNR, which needs no ground truth: power within
±0.1 Hz of the peak and its first harmonic, over everything else in the band.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 6), sharex=True)
rows = []
for ax, (name, sig) in zip(axes, signals.items()):
    r = estimate_bpm(sig, FS, zero_pad=8)
    m = (r.freqs >= 0.5) & (r.freqs <= 4.5)
    ax.plot(r.freqs[m]*60, r.power[m], lw=1.0, color=COL[name])
    ax.axvline(r.bpm, color="k", ls="--", lw=0.9)
    ax.axvspan((r.f_peak-0.1)*60, (r.f_peak+0.1)*60, color=COL[name], alpha=0.15)
    ax.axvspan((2*r.f_peak-0.1)*60, (2*r.f_peak+0.1)*60, color=COL[name], alpha=0.08)
    ax.set_ylabel(name.upper(), color=COL[name]); ax.set_yticks([])
    ax.text(0.99, 0.75, f"{r.bpm:.2f} BPM   SNR {r.snr:+.2f} dB",
            transform=ax.transAxes, ha="right", fontsize=8)
    rows.append((name, r.bpm, r.snr))
axes[0].set_title("spectra, peak marked, shaded = the SNR signal mask (fundamental + 1st harmonic)")
axes[-1].set_xlabel("BPM")
fig.tight_layout()

print(f"{'arm':>6s} {'BPM':>8s} {'SNR dB':>8s}" + (f" {'error':>8s}" if CLIP is None else ""))
for name, bpm, snr in rows:
    err = f" {bpm-TRUE_BPM:+8.2f}" if CLIP is None else ""
    print(f"{name:>6s} {bpm:8.2f} {snr:+8.2f}{err}")


## Stage 6 — sliding window over the whole clip

Window 15 s, hop 1 s, median filter over the last 5 estimates, and an SNR gate
that **rejects** a window rather than emitting a number it cannot support.


In [ ]:
df = analyse_signal(t_raw, rgb_raw, PipelineConfig(fs=FS, window_sec=WINDOW_SEC, hop_sec=1.0))

fig, ax = plt.subplots(figsize=(11, 3.2))
for name in PROJECTIONS:
    sub = df[df.method == name]
    ax.plot(sub.t_center, sub.bpm_smoothed, lw=1.4, color=COL[name], label=name.upper())
    bad = sub[~sub.accepted]
    ax.scatter(bad.t_center, bad.bpm, marker="x", s=22, color=COL[name], alpha=0.7)
if CLIP is None:
    ax.plot(t_raw, reference_bpm, color="k", ls="--", lw=1.2, label="ground truth")
ax.legend(ncol=5, fontsize=8, frameon=False)
ax.set(xlabel="time (s)", ylabel="BPM", title="rolling estimate (× = window rejected by the SNR gate)")
fig.tight_layout()

if CLIP is None:
    print(f"{'arm':>6s} {'MAE':>7s} {'RMSE':>7s} {'SNR':>7s} {'accept':>7s}")
    for name in PROJECTIONS:
        sub = df[df.method == name]
        ref = np.interp(sub.t_center, t_raw, reference_bpm)
        print(f"{name:>6s} {mae(sub.bpm_smoothed, ref):7.2f} {rmse(sub.bpm_smoothed, ref):7.2f} "
              f"{sub.snr.mean():7.2f} {sub.accepted.mean():7.2f}")
    print("\nPearson r is omitted here on purpose: the reference is constant, so its")
    print("standard deviation is zero and r is undefined. See the next cell.")


### Does it *track*, or just sit near the population mean?

The metric above cannot tell the difference — with a constant reference, an arm
that always outputs 72 BPM scores a perfect MAE. This is exactly the trap that
makes resting-HR-only validation worthless, and it is why condition **D
(post-exercise, 100–140 BPM)** is in the protocol.

A ramped heart rate is the smallest experiment that exposes it. Pearson $r$
becomes meaningful the moment the reference actually varies.


In [ ]:
ts_r = nonuniform_timestamps(90.0, FS, jitter_ms=3.0, rng=1)
ramp = synth_rgb(timestamps=ts_r, bpm=(65.0, 125.0), pulse_amplitude=0.004, rng=1)
df_r = analyse_signal(ramp.t, ramp.rgb, PipelineConfig(fs=FS, window_sec=WINDOW_SEC, hop_sec=1.0))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2), gridspec_kw={"width_ratios": [2, 1]})
ax[0].plot(ramp.t, ramp.bpm_true, color="k", ls="--", lw=1.3, label="ground truth")
for name in PROJECTIONS:
    sub = df_r[df_r.method == name]
    ax[0].plot(sub.t_center, sub.bpm_smoothed, lw=1.4, color=COL[name], label=name.upper())
ax[0].legend(ncol=5, fontsize=8, frameon=False)
ax[0].set(xlabel="time (s)", ylabel="BPM", title="HR ramped 65 → 125 BPM")

ref_r = np.interp(df_r[df_r.method == "green"].t_center, ramp.t, ramp.bpm_true)
print("scored against the RAMPED reference:")
print(f"{'arm':>12s} {'MAE':>7s} {'RMSE':>7s} {'r':>7s}")
for name in PROJECTIONS:
    sub = df_r[df_r.method == name]
    ref = np.interp(sub.t_center, ramp.t, ramp.bpm_true)
    print(f"{name:>12s} {mae(sub.bpm_smoothed, ref):7.2f} {rmse(sub.bpm_smoothed, ref):7.2f} "
          f"{pearson(sub.bpm_smoothed, ref):7.3f}")

# The same useless "method" scored both ways.
flat_r = np.full(len(ref_r), 95.0)
ref_const = np.interp(df[df.method == "green"].t_center, t_raw, reference_bpm)
flat_c = np.full(len(ref_const), 72.0)
print(f"{'always 95':>12s} {mae(flat_r, ref_r):7.2f} {rmse(flat_r, ref_r):7.2f} "
      f"{pearson(flat_r, ref_r):7.3f}   <- exposed by a varying reference")
print("\nscored against the CONSTANT 72 BPM reference:")
print(f"{'always 72':>12s} {mae(flat_c, ref_const):7.2f} {rmse(flat_c, ref_const):7.2f} "
      f"{pearson(flat_c, ref_const):7.3f}   <- a perfect score, for a method that measures nothing")

ax[1].scatter(ref_r, df_r[df_r.method == "pos"].bpm_smoothed, s=12, color=COL["pos"], label="POS")
lims = [60, 130]
ax[1].plot(lims, lims, color="k", ls=":", lw=1)
ax[1].set(xlabel="reference BPM", ylabel="estimated BPM", xlim=lims, ylim=lims, title="agreement")
fig.tight_layout()


The same do-nothing predictor scores **0.00 MAE against a constant reference**
and is only exposed once the reference moves. Two conclusions for the
benchmark protocol, both load-bearing:

1. **A resting-HR-only validation cannot distinguish a working method from a
   constant.** Condition D is not optional garnish.
2. **Report Pearson $r$ alongside MAE, never instead of it.** $r$ is undefined
   for a constant predictor and near 1 for one that genuinely tracks — it
   answers a question MAE cannot.


## Where to go next

This notebook establishes that the pipeline recovers a known frequency. It does
**not** establish that it works on faces — that needs ground truth.

1. **`02_benchmark_ubfc.ipynb`** — run `analyse_signal` over every UBFC-rPPG
   subject, align with `rppg.datasets.attach_reference`, produce the 4×4
   MAE grid and a Bland–Altman plot for the winning arm.
2. **`03_ablations.ipynb`** — window-length sweep (MAE vs $T$), detrend
   comparison, skin mask on/off, Welch vs single FFT.
3. **`04_stress_conditions.ipynb`** — the self-collected A/B/C/D set.

A caveat to carry into all three: **SNR measures narrowbandness, not
correctness.** An arm locked onto a periodic motion artefact reports a
confident, high-SNR, wrong number. Re-run the cells above with
`motion_amplitude=0.05, motion_bpm=48` in `synth_rgb` and watch GREEN and ICA
report ~48 BPM with *better* SNR than CHROM. Only ground truth distinguishes
them — which is the entire argument for the benchmark.
